# Adaptive Clinical Recruitment — OpenEnv GRPO Training with Unsloth

**Theme #2: (Super) Long-Horizon Planning & Instruction Following**

This notebook trains a small language model with **GRPO (Group Relative Policy Optimization)** against the live Adaptive Clinical Recruitment environment using **Unsloth** for efficient 4-bit LoRA fine-tuning and **TRL** for the RL training loop.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pratimassaravanan/clinical-recruitment-env/blob/main/notebooks/training_grpo_openenv.ipynb)

**Honest caveats — read before running:**
- This is a **hackathon starter notebook**, not a production training pipeline.
- The live Space serves 3 tasks (`easy_bench`, `medium_bench`, `hard_bench`) with 180 max steps each.
- The action surface matches the current 8 implemented actions.
- Training against a shared remote Space is subject to latency, rate limits, and concurrent-user contention. For serious runs, duplicate the Space or run the environment locally.
- The 0.6B model is intentionally small for fast iteration; real benchmark quality requires larger models and longer training.
- The multi-component reward decomposition below is a principled starting point but the weights will need tuning for your use case.

## 1. Install Dependencies

In [ ]:
!pip install -q unsloth "trl>=0.19.0" "transformers>=4.55.0" "datasets>=2.21.0" "accelerate>=0.34.0" "openenv-core>=0.2.1"

## 2. Configure Environment Access

In [ ]:
ENV_URL = "https://pratimassaravanan-clinical-recruitment.hf.space"
MODEL_NAME = "unsloth/Qwen3-0.6B-unsloth-bnb-4bit"
TASK_SEQUENCE = ["easy_bench", "medium_bench", "hard_bench"]

# Training hyperparameters
LORA_R = 16
LORA_ALPHA = 16
MAX_SEQ_LENGTH = 2048
NUM_GENERATIONS = 2
MAX_COMPLETION_LENGTH = 1024
TRAIN_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 4
NUM_TRAIN_EPOCHS = 1
LEARNING_RATE = 5e-6
OUTPUT_DIR = "grpo_clinical_recruitment"

print(f"Environment: {ENV_URL}")
print(f"Model: {MODEL_NAME}")
print(f"Tasks: {TASK_SEQUENCE}")

## 3. Load Model with Unsloth (4-bit LoRA)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,  # auto-detect
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Trainable parameters: {model.print_trainable_parameters()}")

## 4. Environment Wrapper

The `ClinicalRecruitmentToolEnv` wraps the remote OpenEnv Space with tool-style methods that the TRL `environment_factory` flow expects. Each method corresponds to one of the 8 environment actions.

In [ ]:
from openenv import GenericEnvClient


class ClinicalRecruitmentToolEnv:
    def __init__(self):
        self.client = GenericEnvClient(base_url=ENV_URL).sync()
        self.reward = 0.0
        self.done = False
        self.last_result = None
        self.last_observation = None
        # Track episode history for multi-component reward decomposition
        self.action_history = []
        self.enrollment_history = []
        self.budget_history = []
        self.hypothesis_history = []
        self.initial_budget = None

    def reset(self, **kwargs):
        self.client.connect()
        task = kwargs.get("task") or kwargs.get("task_id") or "easy_bench"
        self.last_result = self.client.reset(task=task)
        self.last_observation = self.last_result.observation
        self.reward = 0.0
        self.done = bool(self.last_result.done)
        # Reset tracking
        self.action_history = []
        self.enrollment_history = []
        self.budget_history = []
        self.hypothesis_history = []
        obs = self.last_observation or {}
        self.initial_budget = obs.get("budget_remaining", 0.0)
        self.enrollment_history.append(obs.get("enrolled_so_far", 0))
        self.budget_history.append(obs.get("budget_remaining", 0.0))
        return self._format_observation()

    def close(self):
        try:
            self.client.close()
        except Exception:
            pass

    def screen_patient(self, patient_id: str, hypothesis: str = "noise_dominant", confidence: float = 0.6) -> str:
        """Screen a candidate patient.

        Args:
            patient_id: Patient id from available_patients
            hypothesis: Current guess about dominant dynamics
            confidence: Confidence in the hypothesis

        Returns:
            Updated environment observation summary.
        """
        return self._step({"action_type": "screen_patient", "patient_id": patient_id, "hypothesis": hypothesis, "confidence": confidence})

    def recontact(self, patient_id: str, hypothesis: str = "noise_dominant", confidence: float = 0.6) -> str:
        """Recontact an eligible patient.

        Args:
            patient_id: Patient id from recontact_candidates
            hypothesis: Current guess about dominant dynamics
            confidence: Confidence in the hypothesis

        Returns:
            Updated environment observation summary.
        """
        return self._step({"action_type": "recontact", "patient_id": patient_id, "hypothesis": hypothesis, "confidence": confidence})

    def allocate_to_site(self, patient_id: str, site_id: str, hypothesis: str = "noise_dominant", confidence: float = 0.6) -> str:
        """Allocate a consented patient to a site.

        Args:
            patient_id: Patient id from allocation_candidates
            site_id: Site id from site_performance
            hypothesis: Current guess about dominant dynamics
            confidence: Confidence in the hypothesis

        Returns:
            Updated environment observation summary.
        """
        return self._step({"action_type": "allocate_to_site", "patient_id": patient_id, "site_id": site_id, "hypothesis": hypothesis, "confidence": confidence})

    def adjust_strategy(self, strategy_change: str, hypothesis: str = "noise_dominant", confidence: float = 0.6) -> str:
        """Adjust recruitment strategy.

        Args:
            strategy_change: Strategy such as increase_outreach or negotiate_site_A
            hypothesis: Current guess about dominant dynamics
            confidence: Confidence in the hypothesis

        Returns:
            Updated environment observation summary.
        """
        return self._step({"action_type": "adjust_strategy", "strategy_change": strategy_change, "hypothesis": hypothesis, "confidence": confidence})

    def plan_next_phase(self, target_phase: str, plan_summary: str = "advance the bottleneck") -> str:
        """Create or revise the current high-level plan.

        Args:
            target_phase: One of screening, conversion, allocation, retention, recovery
            plan_summary: Natural-language summary of the plan

        Returns:
            Updated environment observation summary.
        """
        return self._step({"action_type": "plan_next_phase", "target_phase": target_phase, "plan_summary": plan_summary})

    def summarize_and_index(self, memory_key: str, memory_payload: str) -> str:
        """Write a summary into indexed memory.

        Args:
            memory_key: Key for the indexed memory item
            memory_payload: Summary text to store

        Returns:
            Updated environment observation summary.
        """
        return self._step({"action_type": "summarize_and_index", "memory_key": memory_key, "memory_payload": memory_payload})

    def retrieve_relevant_history(self, memory_query: str) -> str:
        """Retrieve indexed history relevant to the current bottleneck.

        Args:
            memory_query: Query for indexed memory retrieval

        Returns:
            Updated environment observation summary.
        """
        return self._step({"action_type": "retrieve_relevant_history", "memory_query": memory_query})

    def stop_recruitment(self) -> str:
        """End the current episode early.

        Returns:
            Final environment observation summary.
        """
        return self._step({"action_type": "stop_recruitment"})

    def _step(self, action: dict) -> str:
        if self.done:
            raise ValueError("Episode already finished. Call reset().")
        self.last_result = self.client.step(action)
        self.last_observation = self.last_result.observation
        self.reward = float(self.last_result.reward or 0.0)
        self.done = bool(self.last_result.done)
        # Track history for reward decomposition
        self.action_history.append(action.get("action_type", "unknown"))
        obs = self.last_observation or {}
        self.enrollment_history.append(obs.get("enrolled_so_far", 0))
        self.budget_history.append(obs.get("budget_remaining", 0.0))
        hyp = action.get("hypothesis")
        if hyp:
            self.hypothesis_history.append(hyp)
        return self._format_observation()

    def _format_observation(self) -> str:
        obs = self.last_observation or {}
        funnel = obs.get("current_funnel", {})
        return (
            f"step={obs.get('timestamp')} budget={obs.get('budget_remaining')} enrolled={obs.get('enrolled_so_far')}/{obs.get('target_enrollment')} | "
            f"available={len(obs.get('available_patients', []))} recontact={len(obs.get('recontact_candidates', []))} allocate={len(obs.get('allocation_candidates', []))} | "
            f"funnel={funnel} milestone={obs.get('active_milestone', '')} plan={obs.get('current_plan', {})} memory={obs.get('indexed_memory_summary', {})}"
        )

## 5. Multi-Component Reward Functions

Instead of a single scalar reward, we decompose the signal into **5 independent reward components**. Each inspects the environment state and episode history to provide a focused learning signal:

| Component | What it rewards | Range |
|---|---|---|
| `enrollment_progress` | Moving patients through the funnel toward the enrollment target | [0, 1] |
| `budget_efficiency` | Achieving enrollment progress without wasting budget | [0, 1] |
| `screening_accuracy` | Screening patients who actually become eligible/enrolled | [0, 1] |
| `action_diversity` | Using a variety of action types (not just repeating one) | [0, 1] |
| `hypothesis_consistency` | Maintaining a stable hypothesis instead of erratic switching | [0, 1] |

In [ ]:
def reward_enrollment_progress(environments, **kwargs):
    """Reward based on how close enrollment is to the target.
    
    Normalized enrollment fraction: enrolled / target.
    This is the primary outcome signal.
    """
    rewards = []
    for env in environments:
        obs = env.last_observation or {}
        enrolled = obs.get("enrolled_so_far", 0)
        target = obs.get("target_enrollment", 100)
        if target <= 0:
            target = 100
        progress = min(1.0, enrolled / target)
        rewards.append(progress)
    return rewards


def reward_budget_efficiency(environments, **kwargs):
    """Reward efficient use of budget relative to enrollment gained.
    
    High score = lots of enrollment per dollar spent.
    If no budget was spent, returns 0 (no progress made).
    If budget was spent but nothing enrolled, returns 0.
    """
    rewards = []
    for env in environments:
        obs = env.last_observation or {}
        budget_remaining = obs.get("budget_remaining", 0.0)
        initial_budget = env.initial_budget or 1.0
        budget_spent = max(0.0, initial_budget - budget_remaining)
        enrolled = obs.get("enrolled_so_far", 0)
        target = obs.get("target_enrollment", 100)
        if target <= 0:
            target = 100
        if budget_spent < 1.0:
            # No budget spent = no useful work done
            rewards.append(0.0)
        else:
            # Enrollment fraction achieved per fraction of budget used
            enrollment_frac = min(1.0, enrolled / target)
            budget_frac = min(1.0, budget_spent / initial_budget)
            # Efficiency: high enrollment with low budget usage
            if budget_frac > 0:
                efficiency = min(1.0, enrollment_frac / budget_frac)
            else:
                efficiency = 0.0
            rewards.append(efficiency)
    return rewards


def reward_screening_accuracy(environments, **kwargs):
    """Reward based on the screening-to-enrollment conversion rate.
    
    High score = screened patients actually become enrolled (good patient selection).
    Inspects the funnel to compute screened -> enrolled conversion.
    """
    rewards = []
    for env in environments:
        obs = env.last_observation or {}
        funnel = obs.get("current_funnel", {})
        screened = funnel.get("screened", 0)
        enrolled = funnel.get("enrolled", 0)
        dropped = funnel.get("dropped", 0)
        if screened <= 0:
            rewards.append(0.0)
        else:
            # Conversion rate from screened to enrolled, penalized by dropouts
            conversion = enrolled / screened
            dropout_penalty = dropped / screened
            score = max(0.0, min(1.0, conversion - 0.5 * dropout_penalty))
            rewards.append(score)
    return rewards


def reward_action_diversity(environments, **kwargs):
    """Reward using a diverse set of actions rather than repeating one.
    
    Measures the number of unique action types used divided by the total
    number of available action types (8). Encourages the agent to explore
    the full action space: screening, recontact, allocation, strategy,
    planning, memory write, memory read, and stop.
    """
    ALL_ACTION_TYPES = {
        "screen_patient", "recontact", "allocate_to_site", "adjust_strategy",
        "plan_next_phase", "summarize_and_index", "retrieve_relevant_history",
        "stop_recruitment",
    }
    rewards = []
    for env in environments:
        history = getattr(env, "action_history", [])
        if not history:
            rewards.append(0.0)
            continue
        unique_actions = set(history)
        # Fraction of the 8 action types actually used
        diversity = len(unique_actions) / len(ALL_ACTION_TYPES)
        rewards.append(min(1.0, diversity))
    return rewards


def reward_hypothesis_consistency(environments, **kwargs):
    """Reward maintaining a stable hypothesis across the episode.
    
    Penalizes erratic hypothesis switching. 0-1 switches = 1.0,
    2-4 switches = partial credit, 5+ = 0.2.
    Also gives a bonus if the final hypothesis matches the env world_type.
    """
    rewards = []
    for env in environments:
        hypotheses = getattr(env, "hypothesis_history", [])
        if len(hypotheses) < 2:
            rewards.append(0.5)  # neutral if too few hypotheses provided
            continue
        # Count switches
        switches = sum(
            1 for i in range(1, len(hypotheses))
            if hypotheses[i] != hypotheses[i - 1]
        )
        if switches <= 1:
            consistency = 1.0
        elif switches <= 4:
            consistency = 1.0 - (switches - 1) * 0.2
        else:
            consistency = 0.2
        # Bonus for correct final hypothesis
        obs = env.last_observation or {}
        world_type = obs.get("world_type", "unknown")
        hyp_to_world = {
            "dropout_dominant": "dropout",
            "noise_dominant": "noise",
            "site_bias": "site_bias",
        }
        final_hyp = hypotheses[-1] if hypotheses else ""
        if hyp_to_world.get(final_hyp, "") == world_type and world_type:
            accuracy_bonus = 0.2
        else:
            accuracy_bonus = 0.0
        rewards.append(min(1.0, consistency * 0.8 + accuracy_bonus))
    return rewards


# Collect all reward functions into a list for the trainer
REWARD_FUNCS = [
    reward_enrollment_progress,
    reward_budget_efficiency,
    reward_screening_accuracy,
    reward_action_diversity,
    reward_hypothesis_consistency,
]

print(f"Defined {len(REWARD_FUNCS)} independent reward components:")
for fn in REWARD_FUNCS:
    print(f"  - {fn.__name__}")

## 6. Prompt Dataset

We create a small training dataset of system+user prompts, cycling through the three difficulty tasks. Each prompt asks the agent to improve recruitment on the assigned task.

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = """You are a long-horizon clinical trial recruitment agent. Use tools to inspect and improve recruitment progress across screening, recontact, allocation, planning, and memory. Keep the episode moving toward final_score improvement."""

train_dataset = Dataset.from_dict({
    "prompt": [
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Improve recruitment outcomes on the assigned task. Use the environment tools to make progress."}
        ]
        for _ in range(24)
    ],
    "task_id": [TASK_SEQUENCE[i % len(TASK_SEQUENCE)] for i in range(24)],
})

print(f"Training dataset: {len(train_dataset)} prompts")
print(f"Task distribution: {dict((t, sum(1 for x in train_dataset['task_id'] if x == t)) for t in TASK_SEQUENCE)}")

## 7. Before-Training Baseline

Run one episode with the **untrained** (base LoRA) model to establish a baseline score. This lets us measure improvement after training.

In [ ]:
import torch


def run_baseline_episode(model, tokenizer, task="easy_bench", max_actions=20):
    """Run a short episode using greedy generation from the model.
    
    This uses a simple loop: format the observation as a user message,
    generate a response, parse out an action, and step the environment.
    
    Returns:
        dict with episode metrics.
    """
    env = ClinicalRecruitmentToolEnv()
    try:
        obs_text = env.reset(task=task)
    except Exception as e:
        print(f"Failed to connect to environment: {e}")
        return {"task": task, "error": str(e)}

    FastLanguageModel.for_inference(model)
    
    total_reward = 0.0
    actions_taken = 0
    
    for step_i in range(max_actions):
        if env.done:
            break
        
        # Build prompt
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Current state: {obs_text}\n\nDecide the next action. Pick one: screen_patient, recontact, allocate_to_site, adjust_strategy, plan_next_phase, summarize_and_index, retrieve_relevant_history, or stop_recruitment. Respond with the action and parameters."},
        ]
        
        input_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
        inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
                temperature=1.0,
            )
        
        response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        
        # Parse a simple fallback action from the response
        # Try to find a recognized action type in the model output
        action = None
        obs = env.last_observation or {}
        response_lower = response.lower()
        
        if "screen_patient" in response_lower and obs.get("available_patients"):
            pid = obs["available_patients"][0].get("id", "P_0")
            action = {"action_type": "screen_patient", "patient_id": pid, "hypothesis": "noise_dominant", "confidence": 0.6}
        elif "allocate_to_site" in response_lower and obs.get("allocation_candidates") and obs.get("site_performance"):
            pid = obs["allocation_candidates"][0].get("id", "P_0")
            sid = list(obs["site_performance"].keys())[0] if obs["site_performance"] else "site_A"
            action = {"action_type": "allocate_to_site", "patient_id": pid, "site_id": sid, "hypothesis": "noise_dominant", "confidence": 0.6}
        elif "recontact" in response_lower and obs.get("recontact_candidates"):
            pid = obs["recontact_candidates"][0].get("id", "P_0")
            action = {"action_type": "recontact", "patient_id": pid, "hypothesis": "noise_dominant", "confidence": 0.6}
        elif "adjust_strategy" in response_lower:
            action = {"action_type": "adjust_strategy", "strategy_change": "increase_outreach", "hypothesis": "noise_dominant", "confidence": 0.6}
        elif "plan_next_phase" in response_lower:
            action = {"action_type": "plan_next_phase", "target_phase": "screening", "plan_summary": "screen more patients"}
        elif "stop" in response_lower:
            action = {"action_type": "stop_recruitment"}
        else:
            # Default fallback: screen if patients available, else adjust strategy
            if obs.get("available_patients"):
                pid = obs["available_patients"][0].get("id", "P_0")
                action = {"action_type": "screen_patient", "patient_id": pid, "hypothesis": "noise_dominant", "confidence": 0.6}
            else:
                action = {"action_type": "adjust_strategy", "strategy_change": "increase_outreach", "hypothesis": "noise_dominant", "confidence": 0.6}
        
        try:
            obs_text = env._step(action)
            total_reward += env.reward
            actions_taken += 1
        except Exception as e:
            print(f"  Step {step_i} error: {e}")
            break
    
    # Collect final metrics
    final_obs = env.last_observation or {}
    result = {
        "task": task,
        "actions_taken": actions_taken,
        "total_reward": round(total_reward, 4),
        "enrolled": final_obs.get("enrolled_so_far", 0),
        "target": final_obs.get("target_enrollment", 100),
        "budget_remaining": final_obs.get("budget_remaining", 0.0),
        "enrollment_progress": reward_enrollment_progress([env])[0],
        "budget_efficiency": reward_budget_efficiency([env])[0],
        "screening_accuracy": reward_screening_accuracy([env])[0],
        "action_diversity": reward_action_diversity([env])[0],
        "hypothesis_consistency": reward_hypothesis_consistency([env])[0],
    }
    
    env.close()
    return result


print("=" * 60)
print("BEFORE TRAINING - Baseline Episode")
print("=" * 60)
baseline_result = run_baseline_episode(model, tokenizer, task="easy_bench", max_actions=20)
print()
for k, v in baseline_result.items():
    print(f"  {k:>25s}: {v}")
print("=" * 60)

## 8. GRPO Training

Configure and run the GRPO trainer with all 5 reward components. The trainer will:
1. Sample completions from the model for each prompt
2. Execute them against the live environment
3. Score each completion with all 5 reward functions
4. Update the model using group-relative policy optimization

In [ ]:
from trl import GRPOConfig, GRPOTrainer

# Switch model back to training mode
FastLanguageModel.for_training(model)

grpo_config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_COMPLETION_LENGTH,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    learning_rate=LEARNING_RATE,
    logging_steps=1,
    save_steps=25,
    bf16=True,
    optim="adamw_8bit",
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    chat_template_kwargs={"enable_thinking": False},
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    reward_funcs=REWARD_FUNCS,
    environment_factory=ClinicalRecruitmentToolEnv,
    args=grpo_config,
)

print("GRPO trainer configured.")
print(f"  Reward components: {[fn.__name__ for fn in REWARD_FUNCS]}")
print(f"  Output dir: {OUTPUT_DIR}")
print(f"  Epochs: {NUM_TRAIN_EPOCHS}")
print()
print("Starting training...")
print("(This will take a while due to environment interaction latency.)")
print()

trainer.train()

print()
print("Training complete!")

## 9. After-Training Evaluation

Run the same baseline episode with the trained model and compare to the pre-training result.

In [ ]:
print("=" * 60)
print("AFTER TRAINING - Evaluation Episode")
print("=" * 60)
trained_result = run_baseline_episode(model, tokenizer, task="easy_bench", max_actions=20)
print()
for k, v in trained_result.items():
    print(f"  {k:>25s}: {v}")
print("=" * 60)

# Side-by-side comparison
print()
print("=" * 60)
print("BEFORE vs AFTER COMPARISON")
print("=" * 60)
compare_keys = [
    "total_reward", "enrolled", "enrollment_progress",
    "budget_efficiency", "screening_accuracy",
    "action_diversity", "hypothesis_consistency",
]
print(f"  {'Metric':>25s}  {'Before':>10s}  {'After':>10s}  {'Delta':>10s}")
print(f"  {'-'*25}  {'-'*10}  {'-'*10}  {'-'*10}")
for key in compare_keys:
    before = baseline_result.get(key, 0)
    after = trained_result.get(key, 0)
    if isinstance(before, (int, float)) and isinstance(after, (int, float)):
        delta = after - before
        sign = "+" if delta >= 0 else ""
        print(f"  {key:>25s}  {before:>10.4f}  {after:>10.4f}  {sign}{delta:>9.4f}")
    else:
        print(f"  {key:>25s}  {str(before):>10s}  {str(after):>10s}  {'n/a':>10s}")
print("=" * 60)
print()
print("NOTE: With only 1 epoch on 24 prompts and a 0.6B model, large improvements")
print("are not expected. This demonstrates the pipeline works end-to-end.")
print("For real gains, increase epochs, dataset size, and consider a larger model.")

## 10. Save Model

Save the trained LoRA adapter weights, and optionally merge them into a full model for deployment.

In [ ]:
# Save LoRA adapter
LORA_SAVE_PATH = f"{OUTPUT_DIR}/lora_adapter"
model.save_pretrained(LORA_SAVE_PATH)
tokenizer.save_pretrained(LORA_SAVE_PATH)
print(f"LoRA adapter saved to: {LORA_SAVE_PATH}")

# Save merged model (LoRA weights merged into base for standalone inference)
MERGED_SAVE_PATH = f"{OUTPUT_DIR}/merged_model"
merged_model = model.merge_and_unload()
merged_model.save_pretrained(MERGED_SAVE_PATH)
tokenizer.save_pretrained(MERGED_SAVE_PATH)
print(f"Merged model saved to: {MERGED_SAVE_PATH}")
print()
print("To load the merged model later:")
print(f"  from transformers import AutoModelForCausalLM, AutoTokenizer")
print(f"  model = AutoModelForCausalLM.from_pretrained('{MERGED_SAVE_PATH}')")
print(f"  tokenizer = AutoTokenizer.from_pretrained('{MERGED_SAVE_PATH}')")

## Honest Closing Caveats

This notebook demonstrates a **complete end-to-end GRPO training pipeline** for the Adaptive Clinical Recruitment environment, but it is still a hackathon starter:

- **Scale**: 24 prompts and 1 epoch is enough to verify the pipeline works, not enough to produce a well-trained agent. Real training needs hundreds or thousands of episodes.
- **Model size**: The 0.6B parameter model is chosen for speed. Larger models (3B+) will likely perform much better on this long-horizon task.
- **Reward tuning**: The 5 reward components have equal implicit weight via GRPO. In practice, you should experiment with reward scaling, clipping, and relative weighting.
- **Environment contention**: The shared Space at `ENV_URL` may be slow under concurrent load. For serious training, duplicate the Space or run `env.py` locally.
- **Action parsing**: The baseline evaluation uses simple string matching to parse model outputs into actions. The TRL GRPO trainer handles this through its own tool-use flow, but robust action parsing is critical for real deployment.
- **Evaluation rigor**: The before/after comparison runs a single episode on `easy_bench`. Proper evaluation should run multiple episodes across all three tasks and average the results.

Use this as a working foundation, not as evidence of benchmark quality.